# TotalTester -- AI-Powered Functional QA Agent

A proof-of-concept LangGraph agent that generates and executes **browser-based functional tests** from natural-language user stories using Playwright.

### What this notebook demonstrates

1. **Custom Playwright tools** -- LangChain's Playwright toolkit lacks a `fill_text` tool, so we build one
2. **Structured outputs** -- Pydantic schemas for test plans, test results, and evaluator decisions
3. **Multi-node LangGraph** -- Planner → Human Review → Functional Tester → Report Generator → Evaluator
4. **Human-in-the-loop** -- `interrupt_before` pauses the graph so a human can review and approve/modify the test plan
5. **Conditional routing** -- The evaluator can loop back to the planner if testing is incomplete

In [ ]:
from typing import Annotated, List, Any, Optional, Dict
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langchain.agents import Tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from IPython.display import Image, display
from datetime import datetime
from dotenv import load_dotenv
import nest_asyncio
import json
import uuid

nest_asyncio.apply()
load_dotenv(override=True)

## Section 1: Playwright Browser + Custom Tools

LangChain's `PlayWrightBrowserToolkit` provides 7 tools: `navigate_browser`, `click_element`, `extract_text`, `extract_hyperlinks`, `get_elements`, `current_webpage`, and `previous_webpage`.

Two problems:
1. **There is no `fill_text` tool** for typing into form fields
2. **The built-in `click_element` is unreliable** -- it fails on common selectors like `button[type='submit']`

We fix both below by creating custom tools that call Playwright's `page.fill()` and `page.click()` directly.

In [ ]:
async_browser = create_async_playwright_browser(headless=False)
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
playwright_tools = toolkit.get_tools()

print("Built-in Playwright tools:")
for t in playwright_tools:
    print(f"  {t.name}: {t.description[:80]}")


def _get_page():
    """Get the current active page from the async browser."""
    if not async_browser.contexts:
        return None
    context = async_browser.contexts[0]
    return context.pages[-1] if context.pages else None


async def fill_text_fn(input_str: str) -> str:
    """Fill a text field on the current page.
    Input must be a JSON string with 'selector' and 'value' keys.
    """
    page = _get_page()
    if page is None:
        return "Error: no active page. Use navigate_browser first."
    try:
        parsed = json.loads(input_str)
        selector = parsed["selector"]
        value = parsed["value"]
    except (json.JSONDecodeError, KeyError):
        return 'Error: input must be JSON like {"selector": "input#username", "value": "tomsmith"}'
    try:
        await page.fill(selector, value)
        return f"Successfully filled '{selector}' with '{value}'"
    except Exception as e:
        return f"Error filling '{selector}': {e}"


async def click_element_fn(selector: str) -> str:
    """Click an element on the current page using a CSS selector."""
    page = _get_page()
    if page is None:
        return "Error: no active page. Use navigate_browser first."
    try:
        await page.click(selector, timeout=5000)
        return f"Clicked element '{selector}'"
    except Exception as e:
        return f"Error clicking '{selector}': {e}"


fill_tool = Tool(
    name="fill_text",
    func=lambda x: "Use async version",
    coroutine=fill_text_fn,
    description=(
        'Fill a text field on the current page. '
        'Input must be a JSON string with "selector" and "value" keys. '
        'Example: {"selector": "input#username", "value": "tomsmith"}. '
        'Use get_elements first to discover the correct CSS selectors.'
    ),
)

click_tool = Tool(
    name="click_element",
    func=lambda x: "Use async version",
    coroutine=click_element_fn,
    description=(
        "Click on an element on the current page using a CSS selector. "
        "Example: click_element('button[type=\"submit\"]') or click_element('#login-button'). "
        "Use get_elements first to discover the correct CSS selectors."
    ),
)

# Replace the built-in click_element with our custom one, keep the rest
tools = [t for t in playwright_tools if t.name != "click_element"] + [fill_tool, click_tool]
print(f"\nTotal tools available: {len(tools)}")
print("Custom tools replacing/extending the built-in toolkit:")
print(f"  + fill_text: fills form fields via page.fill()")
print(f"  + click_element: clicks elements via page.click() (replaces built-in)")

## Section 2: Tool Demo

Before wiring tools into the graph, let's verify they work. We'll navigate to a login page, discover form selectors, fill them, click submit, and verify the result.

In [ ]:
tool_map = {t.name: t for t in tools}

result = await tool_map["navigate_browser"].ainvoke({"url": "https://the-internet.herokuapp.com/login"})
print("1. Navigate:", result[:120])

result = await tool_map["get_elements"].ainvoke({"selector": "input", "attributes": ["id", "name", "type"]})
print("2. Discover inputs:", result)

result = await tool_map["get_elements"].ainvoke({"selector": "button", "attributes": ["id", "type", "class"]})
print("3. Discover buttons:", result)

result = await tool_map["fill_text"].ainvoke('{"selector": "input#username", "value": "tomsmith"}')
print("4. Fill username:", result)

result = await tool_map["fill_text"].ainvoke('{"selector": "input#password", "value": "SuperSecretPassword!"}')
print("5. Fill password:", result)

result = await tool_map["click_element"].ainvoke({"selector": "button[type='submit']"})
print("6. Click login:", result)

result = await tool_map["extract_text"].ainvoke({})
print("7. Page after login:", result[:200])

## Section 3: Structured Output Schemas and Graph State

We define Pydantic schemas that the LLM must conform to via `with_structured_output()`. This gives us reliable, parseable results from every node.

In [ ]:
class PageTarget(BaseModel):
    url: str = Field(description="Full URL of the page to test")
    description: str = Field(description="Brief description of what this page is")


class ParsedUserStory(BaseModel):
    story: str = Field(description="The user story text")
    test_steps: List[str] = Field(description="Concrete steps to verify this story in a browser")
    expected_outcome: str = Field(description="What a passing test looks like")


class TestPlan(BaseModel):
    pages: List[PageTarget] = Field(description="Pages to test")
    parsed_stories: List[ParsedUserStory] = Field(
        description="User stories broken into browser-executable test steps",
    )
    clarification_needed: bool = Field(
        default=False,
        description="True if more information is needed from the user before testing can begin",
    )
    clarification_question: Optional[str] = Field(
        default=None,
        description="Question to ask the user, if clarification is needed",
    )
    reasoning: str = Field(description="Brief explanation of the test plan decisions")


class FunctionalTestResult(BaseModel):
    test_name: str = Field(description="Name of the test case")
    user_story: str = Field(description="The user story being tested")
    steps_executed: List[str] = Field(description="Steps that were actually performed")
    expected: str = Field(description="Expected outcome")
    actual: str = Field(description="What actually happened")
    status: str = Field(description="pass, fail, or error")
    notes: Optional[str] = Field(default=None, description="Additional observations")


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the completeness and quality of the testing")
    success_criteria_met: bool = Field(description="Whether all user stories have been tested thoroughly")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or the agent is stuck"
    )

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    target_url: str
    user_stories: str
    test_plan: Optional[str]
    plan_approved: bool
    plan_feedback: Optional[str]
    functional_test_results: Optional[str]
    report_path: Optional[str]
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

## Section 4: Node Implementations

Each node is a function that takes the current `State` and returns a partial state update. We set up the LLMs here too.

In [ ]:
planner_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(TestPlan)
worker_llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)
evaluator_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EvaluatorOutput)

### Planner Node

Takes the target URL and user stories, produces a structured `TestPlan` with browser-executable test steps.

In [ ]:
async def planner(state: State) -> Dict[str, Any]:
    stories = state.get("user_stories", "")
    url = state.get("target_url", "")
    plan_feedback = state.get("plan_feedback")

    system_prompt = f"""You are a test planning agent for TotalTester, an AI-powered functional QA tool.
Given a target URL and user stories, produce a structured TestPlan with browser-executable test steps.

Current date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Rules:
- Parse each user story into concrete, browser-executable test steps (click, fill, navigate, assert).
- If no user stories are provided, set clarification_needed to True and ask for them.
- Always include the target URL in pages. Add sub-pages if you can infer them from the stories.
- Each test step should be specific enough for a browser automation agent to execute."""

    user_prompt = f"""Target URL: {url}
User stories:
{stories or "(none provided)"}"""

    if plan_feedback:
        user_prompt += f"\n\nThe human reviewer provided this feedback on your previous plan:\n{plan_feedback}\nPlease revise the plan accordingly."

    result = await planner_llm.ainvoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])

    if result.clarification_needed:
        return {
            "test_plan": result.model_dump_json(),
            "user_input_needed": True,
            "messages": [{"role": "assistant", "content": result.clarification_question or "I need user stories to create a test plan."}],
        }

    return {
        "test_plan": result.model_dump_json(),
        "plan_approved": False,
        "messages": [{"role": "assistant", "content": f"Test plan ready for review:\n\n{result.reasoning}"}],
    }

### Human Review Node

A lightweight pass-through. The graph **interrupts before** this node runs, giving the human a chance to inspect the plan. When the graph resumes, this node checks whether the plan was approved or needs revision.

In [ ]:
async def human_review(state: State) -> Dict[str, Any]:
    if state.get("plan_approved"):
        return {"messages": [{"role": "assistant", "content": "Test plan approved. Starting functional tests..."}]}
    return {
        "plan_feedback": state.get("plan_feedback", "Please revise the plan."),
        "messages": [{"role": "assistant", "content": "Revising test plan based on feedback..."}],
    }

### Functional Tester Node

The worker agent that uses Playwright tools to execute test cases. The system prompt enforces a strict workflow: discover selectors first with `get_elements`, then interact with our custom `fill_text` and `click_element` tools.

In [ ]:
async def functional_tester(state: State) -> Dict[str, Any]:
    system_prompt = f"""You are a QA test execution agent with browser automation tools.

YOUR AVAILABLE TOOLS:
- navigate_browser: go to a URL
- extract_text: read all visible text on the current page
- get_elements: find elements by CSS selector and return their attributes
- click_element: click an element by CSS selector
- fill_text: type into an input field (JSON input: {{"selector": "input#id", "value": "text"}})
- extract_hyperlinks: get all links on the page
- current_webpage: get the current URL

CRITICAL WORKFLOW -- you MUST follow this order for every page interaction:
1. navigate_browser to the page URL.
2. extract_text to read the page and understand its content.
3. get_elements with broad selectors to discover actual element attributes:
   - get_elements("input", ["id", "name", "type", "placeholder"])
   - get_elements("button", ["id", "type", "class", "innerText"])
   - get_elements("a", ["href", "innerText"])
4. ONLY THEN use fill_text or click_element with the exact CSS selectors you discovered.
5. After each action, use extract_text to verify what happened.

SELECTOR RULES:
- click_element input: a CSS selector string, e.g. "button[type='submit']", "#login-button"
- fill_text input: a JSON string, e.g. {{"selector": "input#username", "value": "tomsmith"}}
- NEVER guess selectors. ALWAYS discover them with get_elements first.
- Prefer #id selectors when an id attribute exists. Otherwise use [name='...'] or [type='...'].

Execute the functional test cases from the test plan:
{state["test_plan"]}

Target URL: {state["target_url"]}
Current date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Work through each test case systematically. For each test, report: steps taken, expected vs actual outcome, pass/fail."""

    if state.get("feedback_on_work"):
        system_prompt += f"\n\nPrevious feedback: {state['feedback_on_work']}"

    messages = state["messages"]
    found_system = False
    for msg in messages:
        if isinstance(msg, SystemMessage):
            msg.content = system_prompt
            found_system = True
    if not found_system:
        messages = [SystemMessage(content=system_prompt)] + messages

    response = await worker_llm.ainvoke(messages)
    return {"messages": [response]}

### Report Generator and Evaluator Nodes

In [ ]:
async def report_generator(state: State) -> Dict[str, Any]:
    system_prompt = """You are a test report generator. Consolidate the functional test results into
a clear, well-structured Markdown report with:

1. Executive Summary (pass/fail counts, overall status)
2. Test Results (each user story with its outcome, steps executed, and any notes)
3. Recommendations (next steps, areas of concern)

Use severity indicators and organize by status (failures first). Be concise but thorough."""

    conversation = ""
    for msg in state["messages"]:
        if isinstance(msg, AIMessage) and msg.content:
            conversation += f"Assistant: {msg.content}\n"

    user_prompt = f"""Generate a functional test report for: {state.get("target_url", "unknown")}

Test plan: {state.get("test_plan", "{}")}

Test execution conversation:
{conversation}

Structured results (if available): {state.get("functional_test_results", "{}")}"""

    report_llm = ChatOpenAI(model="gpt-4o-mini")
    result = await report_llm.ainvoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])

    return {
        "report_path": "notebook_output",
        "messages": [{"role": "assistant", "content": f"**Test Report:**\n\n{result.content}"}],
    }

In [ ]:
async def evaluator(state: State) -> Dict[str, Any]:
    plan = json.loads(state.get("test_plan", "{}"))

    system_prompt = """You are a quality evaluator for an AI functional testing agent.
Assess whether the testing was thorough and complete given the test plan.
Consider:
- Were all user stories actually tested?
- Did the agent execute the planned test steps?
- Are the results detailed enough to be actionable?
- Is the report comprehensive?"""

    user_prompt = f"""Test plan: {json.dumps(plan, indent=2)}

Functional results: {state.get("functional_test_results", "none")}
Report path: {state.get("report_path", "none")}"""

    result = await evaluator_llm.ainvoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])

    return {
        "feedback_on_work": result.feedback,
        "success_criteria_met": result.success_criteria_met,
        "user_input_needed": result.user_input_needed,
        "messages": [{"role": "assistant", "content": f"Evaluation: {result.feedback}"}],
    }

### Router Functions

These pure-logic functions decide where the graph goes next at each conditional edge.

In [ ]:
def planner_router(state: State) -> str:
    if state.get("user_input_needed"):
        return "END"
    return "human_review"

def human_review_router(state: State) -> str:
    if state.get("plan_approved"):
        return "functional_tester"
    return "planner"

def functional_tester_router(state: State) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "report_generator"

def evaluator_router(state: State) -> str:
    if state.get("success_criteria_met") or state.get("user_input_needed"):
        return "END"
    return "planner"

## Section 5: Graph Assembly

Wire up nodes and edges, compile with `MemorySaver` for checkpointing and `interrupt_before=["human_review"]` for human-in-the-loop.

In [ ]:
memory = MemorySaver()

builder = StateGraph(State)

builder.add_node("planner", planner)
builder.add_node("human_review", human_review)
builder.add_node("functional_tester", functional_tester)
builder.add_node("tools", ToolNode(tools=tools))
builder.add_node("report_generator", report_generator)
builder.add_node("evaluator", evaluator)

builder.add_edge(START, "planner")
builder.add_conditional_edges("planner", planner_router, {"human_review": "human_review", "END": END})
builder.add_conditional_edges("human_review", human_review_router, {"functional_tester": "functional_tester", "planner": "planner"})
builder.add_conditional_edges("functional_tester", functional_tester_router, {"tools": "tools", "report_generator": "report_generator"})
builder.add_edge("tools", "functional_tester")
builder.add_edge("report_generator", "evaluator")
builder.add_conditional_edges("evaluator", evaluator_router, {"planner": "planner", "END": END})

graph = builder.compile(checkpointer=memory, interrupt_before=["human_review"])

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Section 6: Example -- The Internet Login Page

We run the full graph against a hardcoded URL and user stories. The graph will:
1. Plan the tests (planner node)
2. **Pause** for human review (interrupt_before)
3. We inspect the plan in the next cell
4. We approve and the graph resumes through testing → reporting → evaluation

In [ ]:
TARGET_URL = "https://the-internet.herokuapp.com/login"
USER_STORIES = """As a user I can log in with valid credentials (username: tomsmith, password: SuperSecretPassword!).
As a user I see an error message when I enter invalid credentials."""

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 50}

initial_state = {
    "messages": [HumanMessage(content=f"Run functional tests on {TARGET_URL}")],
    "target_url": TARGET_URL,
    "user_stories": USER_STORIES,
    "test_plan": None,
    "plan_approved": False,
    "plan_feedback": None,
    "functional_test_results": None,
    "report_path": None,
    "feedback_on_work": None,
    "success_criteria_met": False,
    "user_input_needed": False,
}

result = await graph.ainvoke(initial_state, config=config)
print("Graph paused at human_review. The planner has generated a test plan.")

### Human Review: Inspect the Test Plan

This is the human-in-the-loop step. We read the plan from the graph's state and display it. In a production system, this would be a UI; here, you simply read the output and decide.

In [ ]:
state = await graph.aget_state(config)
plan_json = state.values.get("test_plan")

if plan_json:
    plan = json.loads(plan_json)
    print("=" * 60)
    print("TEST PLAN FOR REVIEW")
    print("=" * 60)
    print(f"\nReasoning: {plan.get('reasoning', 'N/A')}\n")
    print("Pages:")
    for p in plan.get("pages", []):
        print(f"  - {p['url']} ({p['description']})")
    print("\nTest Cases:")
    for i, story in enumerate(plan.get("parsed_stories", []), 1):
        print(f"\n  {i}. {story['story']}")
        for step in story.get("test_steps", []):
            print(f"     - {step}")
        print(f"     Expected: {story.get('expected_outcome', 'N/A')}")
    print("\n" + "=" * 60)
    print("Review complete. Run the next cell to approve, or the one after to modify.")
else:
    print("No test plan found in state.")

### Approve the Plan and Resume

We update the graph state with `plan_approved=True` via `aupdate_state`, then resume with `ainvoke(None, config)`. The graph continues through functional testing → report generation → evaluation.

In [ ]:
await graph.aupdate_state(config, {"plan_approved": True})
result = await graph.ainvoke(None, config=config)
print("Graph completed.")

### Results

Display the final messages from the agent -- the test execution, report, and evaluator feedback.

In [ ]:
final_state = await graph.aget_state(config)

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"\nSuccess criteria met: {final_state.values.get('success_criteria_met')}")
print(f"Evaluator feedback: {final_state.values.get('feedback_on_work', 'N/A')}")
print(f"\n--- Last messages ---\n")
for msg in final_state.values.get("messages", [])[-5:]:
    role = "Assistant" if isinstance(msg, AIMessage) else "Human" if isinstance(msg, HumanMessage) else "System"
    content = msg.content if hasattr(msg, "content") else msg.get("content", "")
    if content:
        print(f"[{role}] {content[:500]}")
        print()

## Section 7: Example -- Plan Modification (Human-in-the-Loop)

This demonstrates the **modify** path. We run the planner, review the plan, request changes, see the revised plan, then approve.

In [ ]:
TARGET_URL_2 = "https://demoqa.com/text-box"
USER_STORIES_2 = "As a user I can fill out the text box form with my name and email and see the output."

thread_id_2 = str(uuid.uuid4())
config_2 = {"configurable": {"thread_id": thread_id_2}}

initial_state_2 = {
    "messages": [HumanMessage(content=f"Run functional tests on {TARGET_URL_2}")],
    "target_url": TARGET_URL_2,
    "user_stories": USER_STORIES_2,
    "test_plan": None,
    "plan_approved": False,
    "plan_feedback": None,
    "functional_test_results": None,
    "report_path": None,
    "feedback_on_work": None,
    "success_criteria_met": False,
    "user_input_needed": False,
}

result_2 = await graph.ainvoke(initial_state_2, config=config_2)
print("Graph paused at human_review.")

### Request Plan Modification

Instead of approving, we send feedback requesting an additional test case. The graph routes back to the planner, which revises the plan, and pauses again for review.

In [ ]:
await graph.aupdate_state(config_2, {
    "plan_approved": False,
    "plan_feedback": "Also test that submitting an empty form shows no output or shows validation errors.",
})
result_2 = await graph.ainvoke(None, config=config_2)
print("Planner revised the plan. Graph paused again at human_review.\n")

state_2 = await graph.aget_state(config_2)
plan_json_2 = state_2.values.get("test_plan")
if plan_json_2:
    plan_2 = json.loads(plan_json_2)
    print("REVISED TEST PLAN:")
    print(f"  Reasoning: {plan_2.get('reasoning', 'N/A')}")
    print(f"  Test cases: {len(plan_2.get('parsed_stories', []))}")
    for i, story in enumerate(plan_2.get("parsed_stories", []), 1):
        print(f"    {i}. {story['story']}")

### Approve the Revised Plan and Run

In [ ]:
await graph.aupdate_state(config_2, {"plan_approved": True})
result_2 = await graph.ainvoke(None, config=config_2)

final_state_2 = await graph.aget_state(config_2)
print("=" * 60)
print("EXAMPLE 2 RESULTS")
print("=" * 60)
print(f"\nSuccess criteria met: {final_state_2.values.get('success_criteria_met')}")
print(f"Evaluator feedback: {final_state_2.values.get('feedback_on_work', 'N/A')}")
print(f"\n--- Last messages ---\n")
for msg in final_state_2.values.get("messages", [])[-5:]:
    role = "Assistant" if isinstance(msg, AIMessage) else "Human" if isinstance(msg, HumanMessage) else "System"
    content = msg.content if hasattr(msg, "content") else msg.get("content", "")
    if content:
        print(f"[{role}] {content[:500]}")
        print()

## Section 8: Cleanup

Close the browser when done.

In [ ]:
await async_browser.close()